In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pandas_datareader import data as pdr
from datetime import timedelta

In [ ]:


class DataReader:
  def __init__(self, tickers, start, end, reader_params, ):
    self.tickers = tickers
    self.start = start
    self.end = end
    self.reader_params = reader_params

    self.datastore = self.read_market_data()


  def read_factors(self):
    factors = pdr.DataReader(*self.reader_params)
    factor_data = factors[0]
    self.factor_data = factor_data[self.start:self.end]
    return self.factor_data

  def get_nyse_percentiles(self, l_cutoff, u_cutoff):
    nyse_tickers = []
    bms = []

    for ticker in nyse_tickers:
      t = yf.Ticker(ticker)
      shares = t.info.get("sharesOutstanding")

      history = t.history(start=self.end, end=self.end)['Close']
      market_cap = history.values[-1] * shares
      
      annual_bs = t.balance_sheet
      annual_bv = annual_bs.loc['Stockholders Equity'].iloc[::-1][-1]
      
      bms.append(annual_bv / market_cap)
    bms = np.array(bms)
    
    pl = bms.quantile(l_cutoff)
    ph = bms.quantile(u_cutoff)

    return [pl, ph]


  def read_market_data(self):
    price_data = {}
    BMs = {}

    for ticker in self.tickers:
      t = yf.Ticker(ticker)
      shares = t.info.get("sharesOutstanding")
      history = t.history(start=self.start, end=self.end)['Close']

      market_cap = history.values[-1] * shares
      price_data[("Returns", ticker)] = history.pct_change().values
      
      

      annual_bs = t.balance_sheet
      annual_bv = annual_bs.loc['Stockholders Equity'].iloc[::-1][-1]
      
      BMs[ticker] = annual_bv / market_cap

    BM_i = pd.Series(BMs)

    market_data = pd.DataFrame(price_data)
    
    

    factors = self.read_factors()

    return {
        "market_data": market_data,
        "factors": factors,
        "BM_i": BM_i
    }



In [ ]:
class DataStore(DataReader):
  def __init__(self, tickers, start, end, reader_params, l_cutoff=0.3, u_cutoff=0.7):
    super().__init__(tickers, start, end, reader_params)

    factors = self.datastore["factors"]
    market_data = self.datastore["market_data"]
    self.BM_i = self.datastore["BM_i"]
    self.nyse_percentiles = self.datastore["nyse_percentiles"]

    self.value_split(l_cutoff, u_cutoff)


  def value_split(self, l_cutoff, u_cutoff):
    u_prct = np.percentile(self.BM_i.values, u_cutoff)
    l_prct = np.percentile(self.BM_i.values, l_cutoff)

    self.L = self.BM_i[self.BM_i < l_prct]
    self.M = self.BM_i[(self.BM_i > l_prct) & (self.BM_i < u_prct)]
    self.H = self.BM_i[self.BM_i > u_prct]

  def create_portfolios(self):
    pass

  def construct_factors(self):
    pass

  def _score(self):
    pass


In [ ]:
class FactorModel:
  def __init__(self, n_factors: int=5):
    self.featureset = None
    self.n_factors = n_factors
    self.model = None

  def fit(self):
    pass

In [ ]:
class FactorPremiaModel:
  def __init__(self):
    pass

  def fit(self):
    pass

In [ ]:
class HighRiskPortfolio():
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    self.model = FactorModel()
    self.premia_model = FactorModel()
    self.value_split()
    self.create_portfolios()
    self.construct_factors()
    self._init_model(self.model)
    self._init_factor_premia_model(self.premia_model)
    self.save_model = save_model
    self.model_path = model_path

  def _init_model(self):
    pass

  def _init_factor_premia_model(self):
    pass

  def _fit_factor_premia_model(self):
    pass
  
  def fit(self):
    if self.use_factor_premia_model:
      self._init_factor_premia_model()
      self._fit_factor_premia_model()

    self.model.fit()
    return self.model

  def _save_model(self):
    pass

  def build_portfolio(self):
    self._score()





In [ ]:
class PortfolioConstruction:
  def __init__(self, featureset, save_model: bool = False, model_path:str|None=None):
    pass